<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/4.3%20Using%20Tools%20and%20Memory/3%20Tutorial%20-%20Designing%20a%20Multi%E2%80%91Tool%20Chatbot%20with%20Functions%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install pinned dependencies (Colab-ready; safe to re-run).
# Based on latest compatible versions
!pip install -q langchain-core==1.6.3 langchain-classic==1.0.8 langchain-openai==1.6.2 pydantic==2.13.5


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 4.0 MB/s eta 0:00:00


## Tutorial: Designing a Multi‑Tool Chatbot w/ Functions
In this lesson, we’ll build a tool‑using assistant with function calling and routing logic.



In [5]:
from getpass import getpass
from langchain_openai import ChatOpenAI

# Enter your OpenRouter API key securely when prompted.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# OpenRouter model
MODEL = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    seed=42,
)

print("OpenRouter configured successfully.")
print("Model:", MODEL)

# LangChain tools
from langchain.tools import tool
from langchain_core.tools import Tool
from pydantic import BaseModel, Field



Enter your OpenRouter API key: ··········
OpenRouter configured successfully.
Model: openai/gpt-4o-mini


### Step 1: Define tools + function schemas
We’ll provide a calculator and a weather tool with strict args.


In [6]:
import numexpr as ne
from typing import Literal

class CalcRequest(BaseModel):
    expression: str = Field(..., description="Math expression, e.g., '(12+8)/5'")

@tool("calculator", args_schema=CalcRequest, return_direct=True)
def calculator(expression: str) -> str:
    """
    Evaluate simple math expressions, return a number as string.
    """
    try:
        value = ne.evaluate(expression)
        return str(value.item())
    except Exception as e:
        return f"Error: {e}"

# Backward-compatible alias
calc_tool = calculator

class WeatherRequest(BaseModel):
    city: str = Field(..., description="City name, e.g., 'San Francisco'")
    unit: Literal["celsius", "fahrenheit"] = Field("celsius", description="Temperature unit")

@tool("get_weather", args_schema=WeatherRequest, return_direct=True)
def get_weather(city: str, unit: str = "celsius") -> str:
    """
    Get the weather in a given city.
    """
    sample = {"San Francisco": 18, "New York": 24, "London": 19}
    temp_c = sample.get(city, 20)
    if unit == "fahrenheit":
        temp = round((temp_c * 9/5) + 32)
        return f'{{"city": "{city}", "temp": {temp}, "unit": "F"}}'
    return f'{{"city": "{city}", "temp": {temp_c}, "unit": "C"}}'


### Step 2: Create a function‑calling agent
We’ll let the model decide when to call a tool vs answer directly.


In [8]:
from pprint import pprint

schema = WeatherRequest.model_json_schema()
pprint(schema)


{'properties': {'city': {'description': "City name, e.g., 'San Francisco'",
                         'title': 'City',
                         'type': 'string'},
                'unit': {'default': 'celsius',
                         'description': 'Temperature unit',
                         'enum': ['celsius', 'fahrenheit'],
                         'title': 'Unit',
                         'type': 'string'}},
 'required': ['city'],
 'title': 'WeatherRequest',
 'type': 'object'}


In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when needed."),
    ("human", "What's the temperature in {city} in {unit}?"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent = create_tool_calling_agent(llm, tools=[get_weather], prompt=prompt)
executor = AgentExecutor(agent=agent, tools=[get_weather], verbose=True)

print(executor.invoke({"city": "San Francisco", "unit": "celsius"})["output"])




> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'San Francisco', 'unit': 'celsius'}`


{"city": "San Francisco", "temp": 18, "unit": "C"}


> Finished chain.
{"city": "San Francisco", "temp": 18, "unit": "C"}
